**Text Classification using RNN and LSTM**

##Data Processing

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('spam.csv',encoding='ISO-8859-1')


In [ ]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df['v1'] = df['v1'].map({'spam': 0, 'ham': 1})
df.head()


,v1,v2
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."


##Text Preprocessing

Cleaning by removing punctuation

In [ ]:
import string
string.punctuation
#defining the function to remove punctuation
def remove_punctuation(text):
    punctuationfree="".join([i for i in text if i not in string.punctuation])
    return punctuationfree
#storing the puntuation free text
df['clean_msg']= df['v2'].apply(lambda x:remove_punctuation(x))



Tokenization

In [ ]:
import re
def tokenization(text):
    tokens = re.split('W+',text)
    return tokens
#applying function to the column
df['msg_tokenied']= df['clean_msg'].apply(lambda x: tokenization(x))



Removal of Stop words

In [ ]:
import nltk
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('english')

def remove_stopwords(text):
    output= [i for i in text if i not in stopwords]
    return output

df['no_stopwords']= df['msg_tokenied'].apply(lambda x:remove_stopwords(x))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Porter stemming

In [ ]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()

def stemming(text):
  stem_text = [porter_stemmer.stem(word) for word in text]
  return stem_text

df['msg_stemmed']=df['no_stopwords'].apply(lambda x: stemming(x))

In [ ]:
df.head()

,v1,v2,clean_msg,msg_tokenied,no_stopwords,msg_stemmed
0,1,"Go until jurong point, crazy.. Available only ...",Go until jurong point crazy Available only in ...,[Go until jurong point crazy Available only in...,[Go until jurong point crazy Available only in...,[go until jurong point crazy available only in...
1,1,Ok lar... Joking wif u oni...,Ok lar Joking wif u oni,[Ok lar Joking wif u oni],[Ok lar Joking wif u oni],[ok lar joking wif u oni]
2,0,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...,[Free entry in 2 a wkly comp to win FA Cup fin...,[Free entry in 2 a wkly comp to win FA Cup fin...,[free entry in 2 a wkly comp to win fa cup fin...
3,1,U dun say so early hor... U c already then say...,U dun say so early hor U c already then say,[U dun say so early hor U c already then say],[U dun say so early hor U c already then say],[u dun say so early hor u c already then say]
4,1,"Nah I don't think he goes to usf, he lives aro...",Nah I dont think he goes to usf he lives aroun...,[Nah I dont think he goes to usf he lives arou...,[Nah I dont think he goes to usf he lives arou...,[nah i dont think he goes to usf he lives arou...


##Word Embeddings

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize the tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['msg_stemmed'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['msg_stemmed'])

# Define max length for padding
max_length = 100  # Adjust based on your dataset

# Pad sequences
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Get vocabulary size
vocab_size = len(tokenizer.word_index) + 1


##One Hot Encoding

In [ ]:
from tensorflow.keras.utils import to_categorical

# One-hot encode the sequences
one_hot_encoded_sequences = to_categorical(padded_sequences, num_classes=vocab_size)


##RNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout

# Define the RNN model for one-hot encoded sequences
model_rnn = Sequential([
    Flatten(input_shape=(max_length, vocab_size)),  # Flatten the one-hot encoded input
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


##LSTM Model

In [ ]:
from tensorflow.keras.layers import LSTM, Embedding

# Define the LSTM model with embeddings
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=50, input_length=max_length),  # Use a fixed embedding dimension
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


##Training and Evaluation

###RNN

In [ ]:
# Import the necessary function
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['v1'], test_size=0.2, random_state=42)

# Train the RNN model with embeddings
history_rnn = model_rnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the RNN model
loss_rnn, accuracy_rnn = model_rnn.evaluate(X_test, y_test)
print(f"RNN Accuracy: {accuracy_rnn * 100:.2f}%")

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - accuracy: 0.8572 - loss: 0.4225 - val_accuracy: 0.8621 - val_loss: 0.4078
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.8643 - loss: 0.4015 - val_accuracy: 0.8621 - val_loss: 0.4031
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - accuracy: 0.8640 - loss: 0.4009 - val_accuracy: 0.8621 - val_loss: 0.4030
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.8587 - loss: 0.4102 - val_accuracy: 0.8621 - val_loss: 0.4019
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.8597 - loss: 0.4079 - val_accuracy: 0.8621 - val_loss: 0.4018
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.8638 - loss: 0.3983 - val_accuracy: 0.8621 - val_loss: 0.4021
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.8654 - loss: 0.3965 - val_accuracy: 0.8621 - val_loss: 0.4016
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.8691 - loss: 0.3891 - v

###LSTM

In [ ]:
# Split the data for LSTM model
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['v1'], test_size=0.2, random_state=42)

# Train the LSTM model
history_lstm = model_lstm.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the LSTM model
loss_lstm, accuracy_lstm = model_lstm.evaluate(X_test, y_test)
print(f"LSTM Accuracy: {accuracy_lstm * 100:.2f}%")


Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 20s 130ms/step - accuracy: 0.8441 - loss: 0.4451 - val_accuracy: 0.8621 - val_loss: 0.4036
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 19s 117ms/step - accuracy: 0.8698 - loss: 0.3897 - val_accuracy: 0.8621 - val_loss: 0.4016
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 20s 118ms/step - accuracy: 0.8698 - loss: 0.3879 - val_accuracy: 0.8621 - val_loss: 0.4012
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 20s 116ms/step - accuracy: 0.8716 - loss: 0.3858 - val_accuracy: 0.8621 - val_loss: 0.4033
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 132ms/step - accuracy: 0.8642 - loss: 0.3983 - val_accuracy: 0.8621 - val_loss: 0.4013
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 14s 124ms/step - accuracy: 0.8717 - loss: 0.3835 - val_accuracy: 0.8621 - val_loss: 0.4018
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 13s 117ms/step - accuracy: 0.8636 - loss: 0.3993 - val_accuracy: 0.8621 - val_loss: 0.4012
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 13s 119ms/step - accuracy: 0.8575 - loss: 0